In [1]:
import warnings
warnings.filterwarnings('ignore')

(Cookbook_Saving_hydrophobic_interactions)=
# Saving hydrophobic analyses

*Calculating, naming and persisting typed atom-pair proximity.*

Store independently calculated observations with their source molecular system and retain their geometry, evaluated empty structures and original scientific references. A controlled fragment interface illustrates the workflow without claiming an energetic model.

:::{versionadded} 1.0.0
:::

## Preparing declared chemistry and structures

In [2]:
import molsysmt as msm
from rdkit import Chem
import numpy as np
from molsysmt import pyunitwizard as puw

molsys = msm.convert(Chem.MolFromSmiles('CCC.CCC'), to_form='molsysmt.MolSys')
xyz = np.array([[0,0,0], [.15,0,0], [.30,0,0], [0,.30,0], [.15,.30,0], [.30,.30,0]])
coordinates = np.repeat(xyz[None], 3, axis=0)
coordinates[1, 3:] += [0, 1., 0]
molsys.structures.append(coordinates=puw.quantity(coordinates, 'nm'))

## Calculating and naming the interface

Use disjoint fragment selections. Canonical roles are ordered source indices rather than ligand/protein labels; the evaluation scope retains both selections.

In [3]:
analysis = msm.interactions.hydrophobic.get_hydrophobic_interactions(
    molsys, selection=[0, 1, 2], selection_2=[3, 4, 5], selection_mode='between', pbc=False)
molsys.interactions = {**molsys.interactions, 'fragment_interface': analysis}
assert analysis.evaluated_structure_indices.tolist() == [0, 1, 2]
assert analysis.query(structure_indices=[1]).n_interactions == 0
print(analysis.participant_atoms, analysis.n_interactions)

[1 4] 2


## Saving and loading

Public conversion writes H5MSM 0.5. The round trip retains source axes, relation and occurrence identities, declared scope, producer versions and detached bibliography.

In [4]:
from pathlib import Path
from tempfile import TemporaryDirectory
with TemporaryDirectory() as directory:
    path = str(Path(directory) / 'interface.h5msm')
    msm.convert(molsys, to_form=path)
    loaded = msm.convert(path, to_form='molsysmt.MolSys')
restored = loaded.interactions['fragment_interface']
assert restored.parameters == analysis.parameters
assert restored.software == analysis.software
np.testing.assert_array_equal(restored.to_dict()['occurrence_indices'], analysis.to_dict()['occurrence_indices'])
print(restored.evaluated_structure_indices)

[0 1 2]


## Extracting and reordering

Select original atom and structure indices; extraction remaps all present domains and retains the requested structural order. Removing a participant removes its relation without rerunning detection.

In [5]:
subset = msm.extract(loaded, selection=[4, 1], structure_indices=[2, 0, 1])
assert subset.interactions['fragment_interface'].n_interactions == 2
print(subset.interactions['fragment_interface'].atom_source_indices)
print(subset.interactions['fragment_interface'].structure_source_indices)
removed = msm.extract(loaded, selection=[0, 1, 2])
assert removed.interactions['fragment_interface'].n_interactions == 0
print(removed.interactions['fragment_interface'].evaluated_structure_indices)

[1 4]
[2 0 1]
[0 1 2]


:::{seealso}
:class: dropdown

- {ref}`Tutorial_Get_hydrophobic_interactions` — method, units and periodic geometry.
- {ref}`user-tools-interactions-result` — sparse queries and coverage.
- {ref}`user-foundations-native-world-classes-molsysmt-molsys` — explicit named attachment.
:::